In [5]:
import pandas as pd
from pathlib import Path

In [6]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
save_path = Path("../../data/combined")

In [7]:
sold = pd.read_csv(save_path/"sold.csv")
listings = pd.read_csv(save_path/"listing.csv")
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

C:\Users\alext\AppData\Local\Temp\ipykernel_25516\3142215104.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: ListAgentEmail, 3: FireplaceYN, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(save_path/"sold.csv")
C:\Users\alext\AppData\Local\Temp\ipykernel_25516\3142215104.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: ElementarySchool, 2: BuyerAgencyCompensationType, 3: MiddleOrJuniorSchool, 4: FireplaceYN) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv(save_path/"listing.csv")


In [8]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')['rate_30yr_fixed']
        .mean()
        .reset_index()
)

In [9]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

In [10]:
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')

In [11]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [14]:
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [18]:
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-30    2024-01    858000.0           6.6425
4  2024-01-29    2024-01   1890500.0           6.6425


In [19]:
listings_with_rates.to_csv(save_path/"listing_fred.csv", encoding="utf-8", index=False)
sold_with_rates.to_csv(save_path/"sold_fred.csv", encoding="utf-8", index=False)